In [0]:
hours_to_backfill = ['2026-02-02T15:00:00', '2026-02-19T22:00:00']

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.viewing_content_golden_temp_test_backfill;
CREATE TABLE dev.mohit_gangwani.viewing_content_golden_temp_test_backfill AS
SELECT * FROM prod.detection.viewing_content_golden
WHERE session_start_hour IN ('2026-02-02 15:00:00', '2026-02-19 22:00:00')

In [0]:
def get_sql_for_backfill(hour_to_backfill):
    sql = f"""
        MERGE INTO dev.mohit_gangwani.viewing_content_golden_temp_test_backfill AS target
    USING (
        WITH inscape_map_deduped AS (
            SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id, channel_affiliate
            FROM (
                SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id
                , CASE WHEN st.inscape_station_name IS NOT NULL THEN st.inscape_station_name
                    WHEN LOWER(st.station_affil) LIKE '%affiliate%'
                        OR LOWER(st.station_affil) LIKE '%independent%'
                        OR LOWER(st.station_affil) LIKE '%low power%' THEN st.station_affil
                END AS channel_affiliate
                , ROW_NUMBER() OVER (PARTITION BY ism.mapped_vendor, ism.mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
                FROM prod.detection.inscape_station_map ism
                JOIN prod.detection.epg_station st
                ON st.station_id = ism.mapped_vendor_station_id
                AND st.vendor_name = ism.mapped_vendor
            ) ism
            WHERE ism.rn = 1
        )
        SELECT 
            c.fk_tvid
            , c.session_start
            , c.session_end
            , CASE WHEN c.tuner_channel_number IS NOT NULL then 'LINEAR'  
                ELSE c.content_type
            END AS tuner_content_type 
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_show.database_key END  AS tuner_tms_episode_id
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_show.database_key END AS tuner_tivo_episode_id
            , REGEXP_REPLACE(CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_show.title END, '[\“\”\"\^\@\,]', '') AS tuner_tms_title
            , REGEXP_REPLACE(CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_show.title END, '[\“\”\"\^\@\,]', '') AS tuner_tivo_title
            , CASE WHEN c.file_ingested THEN NULL ELSE c.tms_tuner_airdate END AS tuner_tms_airdate
            , CASE WHEN c.file_ingested THEN NULL ELSE c.tuner_airdate     END AS tuner_tivo_airdate
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_map.inscape_call_sign  END AS tuner_tms_channel_callsign
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_map.inscape_call_sign END AS tuner_tivo_channel_callsign
            , CASE WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL
                        THEN LEAST((unix_timestamp(c.session_start)-unix_timestamp(COALESCE(c.airdate, c.tms_airdate))), c.runtime) 
                ELSE LEAST(c.media_time_start, c.runtime)
            END AS tuner_mt_start
            , tuner_tms_map.channel_affiliate  AS tuner_tms_channel_affiliate
            , tuner_tivo_map.channel_affiliate AS tuner_tivo_channel_affiliate
            , CASE WHEN COALESCE(c.tuner_channel_id, NULLIF(c.tms_tuner_channel_id,98989898)) IS NOT NULL THEN 't'
                WHEN c.is_live = TRUE THEN 't'
                WHEN c.is_live = FALSE THEN 'f'
            END AS tuner_is_live
            , c.tuner_channel_number
            , NULLIF(REGEXP_REPLACE(tuner_tms_show.genre, ', ?', '|'), '') AS tuner_tms_show_genre
            , NULLIF(REGEXP_REPLACE(tuner_tivo_show.genre, ', ?', '|'), '') AS tuner_tivo_show_genre
        FROM prod.detection.viewing_content_firehose AS c
        LEFT OUTER JOIN inscape_map_deduped AS tuner_tivo_map
            ON tuner_tivo_map.mapped_vendor_station_id = c.tuner_channel_id
        AND tuner_tivo_map.mapped_vendor = 'TIVO'
        LEFT OUTER JOIN prod.detection.epg_show AS tuner_tivo_show
            ON tuner_tivo_show.show_id = c.tuner_program_id
        AND tuner_tivo_show.vendor_name = 'TIVO'
        LEFT OUTER JOIN prod.detection.epg_show AS tuner_tms_show
            ON tuner_tms_show.show_id = c.tms_tuner_program_id
        AND tuner_tms_show.vendor_name = 'TMS'
        LEFT OUTER JOIN inscape_map_deduped AS tuner_tms_map
            ON tuner_tms_map.mapped_vendor_station_id = c.tms_tuner_channel_id
        AND tuner_tms_map.mapped_vendor = 'TMS'
        WHERE c.partition_key = DATE('{hour_to_backfill}'::TIMESTAMP)
          AND c.session_start >= '{hour_to_backfill}'::TIMESTAMP
          AND c.session_start < TIMESTAMPADD(HOUR, 1, '{hour_to_backfill}')
          AND COALESCE(c.tuner_program_id, c.tms_tuner_program_id, c.tms_tuner_channel_id, c.tuner_channel_id) IS NOT NULL
    ) AS source
    ON target.fk_tvid = source.fk_tvid
    AND target.session_start = source.session_start
    AND target.session_end = source.session_end
    AND target.session_start_hour = '{hour_to_backfill}'::TIMESTAMP
    AND target.session_start >= '{hour_to_backfill}'::TIMESTAMP
    AND target.session_start < TIMESTAMPADD(HOUR, 1, '{hour_to_backfill}'::TIMESTAMP)
    AND (target.tuner_tms_episode_id IS NULL
        OR target.tuner_tivo_episode_id IS NULL
        OR target.tuner_tms_title IS NULL
        OR target.tuner_tivo_title IS NULL
        OR target.tuner_tms_airdate IS NULL
        OR target.tuner_tivo_airdate IS NULL
        OR target.tuner_tms_channel_callsign IS NULL
        OR target.tuner_tivo_channel_callsign IS NULL)
    WHEN MATCHED THEN UPDATE SET
        target.tuner_content_type = source.tuner_content_type,
        target.tuner_tms_episode_id = source.tuner_tms_episode_id,
        target.tuner_tivo_episode_id = source.tuner_tivo_episode_id,
        target.tuner_tms_title = source.tuner_tms_title,
        target.tuner_tivo_title = source.tuner_tivo_title,
        target.tuner_tms_airdate = source.tuner_tms_airdate,
        target.tuner_tivo_airdate = source.tuner_tivo_airdate,
        target.tuner_tms_channel_callsign = source.tuner_tms_channel_callsign,
        target.tuner_tivo_channel_callsign = source.tuner_tivo_channel_callsign,
        target.tuner_mt_start = source.tuner_mt_start,
        target.tuner_tms_channel_affiliate = source.tuner_tms_channel_affiliate,
        target.tuner_tivo_channel_affiliate = source.tuner_tivo_channel_affiliate,
        target.tuner_is_live = source.tuner_is_live,
        target.tuner_channel_number = source.tuner_channel_number,
        target.tuner_tms_show_genre = source.tuner_tms_show_genre,
        target.tuner_tivo_show_genre = source.tuner_tivo_show_genre
        """
    return sql

In [0]:
for hour_to_backfill in hours_to_backfill:
    print(get_sql_for_backfill(hour_to_backfill))

In [0]:
%sql

        MERGE INTO dev.mohit_gangwani.viewing_content_golden_temp_test_backfill AS target
    USING (
        WITH inscape_map_deduped AS (
            SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id, channel_affiliate
            FROM (
                SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id
                , CASE WHEN st.inscape_station_name IS NOT NULL THEN st.inscape_station_name
                    WHEN LOWER(st.station_affil) LIKE '%affiliate%'
                        OR LOWER(st.station_affil) LIKE '%independent%'
                        OR LOWER(st.station_affil) LIKE '%low power%' THEN st.station_affil
                END AS channel_affiliate
                , ROW_NUMBER() OVER (PARTITION BY ism.mapped_vendor, ism.mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
                FROM prod.detection.inscape_station_map ism
                JOIN prod.detection.epg_station st
                ON st.station_id = ism.mapped_vendor_station_id
                AND st.vendor_name = ism.mapped_vendor
            ) ism
            WHERE ism.rn = 1
        )
        SELECT 
            c.fk_tvid
            , c.session_start
            , c.session_end
            , CASE WHEN c.tuner_channel_number IS NOT NULL then 'LINEAR'  
                ELSE c.content_type
            END AS tuner_content_type 
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_show.database_key END  AS tuner_tms_episode_id
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_show.database_key END AS tuner_tivo_episode_id
            , REGEXP_REPLACE(CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_show.title END, '[\“\”"\^\@\,]', '') AS tuner_tms_title
            , REGEXP_REPLACE(CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_show.title END, '[\“\”"\^\@\,]', '') AS tuner_tivo_title
            , CASE WHEN c.file_ingested THEN NULL ELSE c.tms_tuner_airdate END AS tuner_tms_airdate
            , CASE WHEN c.file_ingested THEN NULL ELSE c.tuner_airdate     END AS tuner_tivo_airdate
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_map.inscape_call_sign  END AS tuner_tms_channel_callsign
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_map.inscape_call_sign END AS tuner_tivo_channel_callsign
            , CASE WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL
                        THEN LEAST((unix_timestamp(c.session_start)-unix_timestamp(COALESCE(c.airdate, c.tms_airdate))), c.runtime) 
                ELSE LEAST(c.media_time_start, c.runtime)
            END AS tuner_mt_start
            , tuner_tms_map.channel_affiliate  AS tuner_tms_channel_affiliate
            , tuner_tivo_map.channel_affiliate AS tuner_tivo_channel_affiliate
            , CASE WHEN COALESCE(c.tuner_channel_id, NULLIF(c.tms_tuner_channel_id,98989898)) IS NOT NULL THEN 't'
                WHEN c.is_live = TRUE THEN 't'
                WHEN c.is_live = FALSE THEN 'f'
            END AS tuner_is_live
            , c.tuner_channel_number
            , NULLIF(REGEXP_REPLACE(tuner_tms_show.genre, ', ?', '|'), '') AS tuner_tms_show_genre
            , NULLIF(REGEXP_REPLACE(tuner_tivo_show.genre, ', ?', '|'), '') AS tuner_tivo_show_genre
        FROM prod.detection.viewing_content_firehose AS c
        LEFT OUTER JOIN inscape_map_deduped AS tuner_tivo_map
            ON tuner_tivo_map.mapped_vendor_station_id = c.tuner_channel_id
        AND tuner_tivo_map.mapped_vendor = 'TIVO'
        LEFT OUTER JOIN prod.detection.epg_show AS tuner_tivo_show
            ON tuner_tivo_show.show_id = c.tuner_program_id
        AND tuner_tivo_show.vendor_name = 'TIVO'
        LEFT OUTER JOIN prod.detection.epg_show AS tuner_tms_show
            ON tuner_tms_show.show_id = c.tms_tuner_program_id
        AND tuner_tms_show.vendor_name = 'TMS'
        LEFT OUTER JOIN inscape_map_deduped AS tuner_tms_map
            ON tuner_tms_map.mapped_vendor_station_id = c.tms_tuner_channel_id
        AND tuner_tms_map.mapped_vendor = 'TMS'
        WHERE c.partition_key = DATE('2026-02-02T15:00:00'::TIMESTAMP)
          AND c.session_start >= '2026-02-02T15:00:00'::TIMESTAMP
          AND c.session_start < TIMESTAMPADD(HOUR, 1, '2026-02-02T15:00:00')
          AND COALESCE(c.tuner_program_id, c.tms_tuner_program_id, c.tms_tuner_channel_id, c.tuner_channel_id) IS NOT NULL
    ) AS source
    ON target.fk_tvid = source.fk_tvid
    AND target.session_start = source.session_start
    AND target.session_end = source.session_end
    AND target.session_start_hour = '2026-02-02T15:00:00'::TIMESTAMP
    AND target.session_start >= '2026-02-02T15:00:00'::TIMESTAMP
    AND target.session_start < TIMESTAMPADD(HOUR, 1, '2026-02-02T15:00:00'::TIMESTAMP)
    AND (target.tuner_tms_episode_id IS NULL
        OR target.tuner_tivo_episode_id IS NULL
        OR target.tuner_tms_title IS NULL
        OR target.tuner_tivo_title IS NULL
        OR target.tuner_tms_airdate IS NULL
        OR target.tuner_tivo_airdate IS NULL
        OR target.tuner_tms_channel_callsign IS NULL
        OR target.tuner_tivo_channel_callsign IS NULL)
    WHEN MATCHED THEN UPDATE SET
        target.tuner_content_type = source.tuner_content_type,
        target.tuner_tms_episode_id = source.tuner_tms_episode_id,
        target.tuner_tivo_episode_id = source.tuner_tivo_episode_id,
        target.tuner_tms_title = source.tuner_tms_title,
        target.tuner_tivo_title = source.tuner_tivo_title,
        target.tuner_tms_airdate = source.tuner_tms_airdate,
        target.tuner_tivo_airdate = source.tuner_tivo_airdate,
        target.tuner_tms_channel_callsign = source.tuner_tms_channel_callsign,
        target.tuner_tivo_channel_callsign = source.tuner_tivo_channel_callsign,
        target.tuner_mt_start = source.tuner_mt_start,
        target.tuner_tms_channel_affiliate = source.tuner_tms_channel_affiliate,
        target.tuner_tivo_channel_affiliate = source.tuner_tivo_channel_affiliate,
        target.tuner_is_live = source.tuner_is_live,
        target.tuner_channel_number = source.tuner_channel_number,
        target.tuner_tms_show_genre = source.tuner_tms_show_genre,
        target.tuner_tivo_show_genre = source.tuner_tivo_show_genre;

In [0]:
%sql

        

        MERGE INTO dev.mohit_gangwani.viewing_content_golden_temp_test_backfill AS target
    USING (
        WITH inscape_map_deduped AS (
            SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id, channel_affiliate
            FROM (
                SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id
                , CASE WHEN st.inscape_station_name IS NOT NULL THEN st.inscape_station_name
                    WHEN LOWER(st.station_affil) LIKE '%affiliate%'
                        OR LOWER(st.station_affil) LIKE '%independent%'
                        OR LOWER(st.station_affil) LIKE '%low power%' THEN st.station_affil
                END AS channel_affiliate
                , ROW_NUMBER() OVER (PARTITION BY ism.mapped_vendor, ism.mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
                FROM prod.detection.inscape_station_map ism
                JOIN prod.detection.epg_station st
                ON st.station_id = ism.mapped_vendor_station_id
                AND st.vendor_name = ism.mapped_vendor
            ) ism
            WHERE ism.rn = 1
        )
        SELECT 
            c.fk_tvid
            , c.session_start
            , c.session_end
            , CASE WHEN c.tuner_channel_number IS NOT NULL then 'LINEAR'  
                ELSE c.content_type
            END AS tuner_content_type 
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_show.database_key END  AS tuner_tms_episode_id
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_show.database_key END AS tuner_tivo_episode_id
            , REGEXP_REPLACE(CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_show.title END, '[\“\”"\^\@\,]', '') AS tuner_tms_title
            , REGEXP_REPLACE(CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_show.title END, '[\“\”"\^\@\,]', '') AS tuner_tivo_title
            , CASE WHEN c.file_ingested THEN NULL ELSE c.tms_tuner_airdate END AS tuner_tms_airdate
            , CASE WHEN c.file_ingested THEN NULL ELSE c.tuner_airdate     END AS tuner_tivo_airdate
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_map.inscape_call_sign  END AS tuner_tms_channel_callsign
            , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_map.inscape_call_sign END AS tuner_tivo_channel_callsign
            , CASE WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL
                        THEN LEAST((unix_timestamp(c.session_start)-unix_timestamp(COALESCE(c.airdate, c.tms_airdate))), c.runtime) 
                ELSE LEAST(c.media_time_start, c.runtime)
            END AS tuner_mt_start
            , tuner_tms_map.channel_affiliate  AS tuner_tms_channel_affiliate
            , tuner_tivo_map.channel_affiliate AS tuner_tivo_channel_affiliate
            , CASE WHEN COALESCE(c.tuner_channel_id, NULLIF(c.tms_tuner_channel_id,98989898)) IS NOT NULL THEN 't'
                WHEN c.is_live = TRUE THEN 't'
                WHEN c.is_live = FALSE THEN 'f'
            END AS tuner_is_live
            , c.tuner_channel_number
            , NULLIF(REGEXP_REPLACE(tuner_tms_show.genre, ', ?', '|'), '') AS tuner_tms_show_genre
            , NULLIF(REGEXP_REPLACE(tuner_tivo_show.genre, ', ?', '|'), '') AS tuner_tivo_show_genre
        FROM prod.detection.viewing_content_firehose AS c
        LEFT OUTER JOIN inscape_map_deduped AS tuner_tivo_map
            ON tuner_tivo_map.mapped_vendor_station_id = c.tuner_channel_id
        AND tuner_tivo_map.mapped_vendor = 'TIVO'
        LEFT OUTER JOIN prod.detection.epg_show AS tuner_tivo_show
            ON tuner_tivo_show.show_id = c.tuner_program_id
        AND tuner_tivo_show.vendor_name = 'TIVO'
        LEFT OUTER JOIN prod.detection.epg_show AS tuner_tms_show
            ON tuner_tms_show.show_id = c.tms_tuner_program_id
        AND tuner_tms_show.vendor_name = 'TMS'
        LEFT OUTER JOIN inscape_map_deduped AS tuner_tms_map
            ON tuner_tms_map.mapped_vendor_station_id = c.tms_tuner_channel_id
        AND tuner_tms_map.mapped_vendor = 'TMS'
        WHERE c.partition_key = DATE('2026-02-19T22:00:00'::TIMESTAMP)
          AND c.session_start >= '2026-02-19T22:00:00'::TIMESTAMP
          AND c.session_start < TIMESTAMPADD(HOUR, 1, '2026-02-19T22:00:00')
          AND COALESCE(c.tuner_program_id, c.tms_tuner_program_id, c.tms_tuner_channel_id, c.tuner_channel_id) IS NOT NULL
    ) AS source
    ON target.fk_tvid = source.fk_tvid
    AND target.session_start = source.session_start
    AND target.session_end = source.session_end
    AND target.session_start_hour = '2026-02-19T22:00:00'::TIMESTAMP
    AND target.session_start >= '2026-02-19T22:00:00'::TIMESTAMP
    AND target.session_start < TIMESTAMPADD(HOUR, 1, '2026-02-19T22:00:00'::TIMESTAMP)
    AND (target.tuner_tms_episode_id IS NULL
        OR target.tuner_tivo_episode_id IS NULL
        OR target.tuner_tms_title IS NULL
        OR target.tuner_tivo_title IS NULL
        OR target.tuner_tms_airdate IS NULL
        OR target.tuner_tivo_airdate IS NULL
        OR target.tuner_tms_channel_callsign IS NULL
        OR target.tuner_tivo_channel_callsign IS NULL)
    WHEN MATCHED THEN UPDATE SET
        target.tuner_content_type = source.tuner_content_type,
        target.tuner_tms_episode_id = source.tuner_tms_episode_id,
        target.tuner_tivo_episode_id = source.tuner_tivo_episode_id,
        target.tuner_tms_title = source.tuner_tms_title,
        target.tuner_tivo_title = source.tuner_tivo_title,
        target.tuner_tms_airdate = source.tuner_tms_airdate,
        target.tuner_tivo_airdate = source.tuner_tivo_airdate,
        target.tuner_tms_channel_callsign = source.tuner_tms_channel_callsign,
        target.tuner_tivo_channel_callsign = source.tuner_tivo_channel_callsign,
        target.tuner_mt_start = source.tuner_mt_start,
        target.tuner_tms_channel_affiliate = source.tuner_tms_channel_affiliate,
        target.tuner_tivo_channel_affiliate = source.tuner_tivo_channel_affiliate,
        target.tuner_is_live = source.tuner_is_live,
        target.tuner_channel_number = source.tuner_channel_number,
        target.tuner_tms_show_genre = source.tuner_tms_show_genre,
        target.tuner_tivo_show_genre = source.tuner_tivo_show_genre;